# Semantic Information Retrieval

This notebook shows how medical knowledge can be used to improve an Information Retrieval (IR) system. Working with the **TREC-COVID** and **PubMed** datasets, we implement and compare two types of semantic techniques:

**Query & Document Expansion** — Expanding queries or documents with related medical terms to better match relevant results:
- Query Expansion via Pseudo Relevance Feedback (PRF)
- Query Expansion with MeSH terms
- Document Expansion with MeSH terms

**Relevance Ranking** — Going beyond simple keyword matching by using semantic signals to rank results:
- Weighted Relevance Ranking
- Learning to Rank (LTR)
- Learning to Rank with Semantic Features

## 1. Data Preparation

Before building the retrieval models, we need to attach medical metadata to our documents. We use the [Entrez E-utilities API](https://www.ncbi.nlm.nih.gov/books/NBK25500/) to fetch **MeSH (Medical Subject Headings)** terms for each paper in the **PubMed** database.

MeSH is a structured medical vocabulary maintained by the U.S. National Library of Medicine. Tagging documents and queries with MeSH terms lets the system match based on medical concepts, not just raw words — which makes a big difference in biomedical search.

In [3]:
from google.colab import files
import pandas as pd
import urllib.request
import json
from bs4 import BeautifulSoup
from collections import Counter

## 1.1 Fetching MeSH Terms from PubMed

We retrieve MeSH terms for our document collection in three steps:

1. Load `covid_df.csv` and extract the PubMed paper IDs into `str_ids`
2. Call the `efetch` endpoint of the E-utilities API to pull document metadata
3. Parse the response and store the MeSH terms in `meshheadings_all`

In [4]:
covid_df = files.upload()

Saving covid_df.csv to covid_df.csv


In [5]:
covid_df = pd.read_csv("covid_df.csv")

The E-utilities API expects document IDs as a single comma-separated string. We build this from our dataframe and pass it directly to the `efetch` endpoint.

In [6]:
list_ids =covid_df['doc_id'].tolist()
str_ids=','.join(str(id) for id in list_ids)

In [7]:
str_ids

'37972180,37972171,37943961,37943932,37943911,37883560,37824671,37824661,37797027,37797016,37769081,37769070,37676938,37616383,37471556,37471538,37410823,37384694,37384688,37289896,37289891,37289888,37228211,37228209,37167400,37141365,37023197,37023187,36996211,36952419,36952417,36926981,36795833,36795816,36758089,36730407,36730404,36730402,36730399,36656953,36634184,36603077,36538032,36520904,36480629,36480617,36454863,36454820,36423297,36423266,36423264,36395209,36356153,36356121,36302057,36264829,36264802,36228020,36227990,36201592,36201569,36173850,36173845,36137017,36108049,36108021,36074854,36074833,36074821,36007042,36007033,35981045,35981036,35981032,35926025,35881010,35881005,35857703,35857620,35857614,35857562,35857529,35857439,35771934,35762884,35737812,35737809,35709281,35699621,35679395,35653468,35617418,35617411,35608456,35608440,35587986,35587962,35549399,35511985,35511944,35482858,35446660,35420947,35389793,35324314,35324284,35324257,35289632,35271343,35271341,35271333,

In [8]:
title_id=covid_df.loc[covid_df['doc_id']==37972180,'Title'].values[0]
print(title_id)

Humans are biocultural, science should be too.


In [9]:
# Build the API request URL with comma-separated PubMed IDs
# Note: E-utilities allows a maximum of 3 requests per second
articles_api_url = f'http://eutils.ncbi.nlm.nih.gov/entrez//eutils/efetch.fcgi?db=pubmed&id={str_ids}'
articles_list = urllib.request.urlopen(articles_api_url).read().decode('utf-8')

In [10]:
# Parse the XML response with BeautifulSoup and isolate PubmedArticle entries
articles_bs = BeautifulSoup(articles_list,features="xml")
articles_iterable = articles_bs.find_all('PubmedArticle')


# Extract MeSH terms for each article and store them by PubMed ID
meshheadings_all = {}
n_articles_without_mesh = 0

for iter, article in enumerate(articles_iterable):
  uid = article.find('PMID').text
  mesh = article.select('MeshHeadingList')
  if len(mesh) > 0:
    mesh_xml = article.find('MeshHeadingList').find_all('MeshHeading')
    mesh_terms = [ x.text for x in mesh_xml ]
    meshheadings_all[uid] = mesh_terms
  else:
    n_articles_without_mesh +=1
    print(f"article number {[uid]} is without mesh terms")

print("mesh terms - example: ", meshheadings_all['35881005'])

article number ['37943961'] is without mesh terms
article number ['36454863'] is without mesh terms
article number ['36356153'] is without mesh terms
article number ['36137017'] is without mesh terms
article number ['35709281'] is without mesh terms
article number ['35324284'] is without mesh terms
article number ['35239375'] is without mesh terms
article number ['35040668'] is without mesh terms
article number ['34941393'] is without mesh terms
article number ['34941386'] is without mesh terms
article number ['34822275'] is without mesh terms
article number ['34762477'] is without mesh terms
article number ['34735221'] is without mesh terms
article number ['34672761'] is without mesh terms
article number ['34643114'] is without mesh terms
article number ['34591641'] is without mesh terms
article number ['34516819'] is without mesh terms
article number ['34516784'] is without mesh terms
article number ['34016743'] is without mesh terms
article number ['33931567'] is without mesh terms


Let's get a quick overview of the extracted MeSH terms: total count, number of unique terms, and distribution across documents.


In [11]:
# Terms per document

print("number of document containing MeSH term:",len(meshheadings_all))
print("number of document without MeSH term:", n_articles_without_mesh)
terms_per_docs = {uid: len(mesh_terms) for uid, mesh_terms in meshheadings_all.items()}
print("MeSH_terms were extracted per document:",terms_per_docs)

# Total and unique term counts
total_terms=sum((len(mesh_terms)) for mesh_terms in meshheadings_all.values())
print("MeSH_terms were extracted totally:",total_terms)
flattened_meshheading_all=[mesh_term for mesh_terms in meshheadings_all.values() for mesh_term in mesh_terms]
unique_terms=Counter(flattened_meshheading_all)
print("unique MeSH_terms are:",unique_terms)
print("number of unique MeSH_terms is:", len(unique_terms))

number of document containing MeSH term: 306
number of document without MeSH term: 21
MeSH_terms were extracted per document: {'37972180': 5, '37943932': 9, '37943911': 7, '37883560': 12, '37824671': 7, '37824661': 4, '37797027': 11, '37797016': 7, '37769081': 6, '37616383': 7, '37471556': 3, '37471538': 6, '37410823': 6, '37384694': 6, '37384688': 3, '37289896': 6, '37289891': 7, '37289888': 8, '37228211': 6, '37228209': 8, '37167400': 5, '37141365': 5, '37023197': 6, '37023187': 5, '36996211': 8, '36952419': 4, '36952417': 10, '36926981': 8, '36795833': 9, '36795816': 7, '36758089': 10, '36730407': 8, '36730404': 6, '36730402': 8, '36730399': 5, '36656953': 7, '36603077': 5, '36538032': 10, '36520904': 7, '36480617': 4, '36454820': 5, '36423297': 9, '36423266': 5, '36423264': 10, '36395209': 6, '36356121': 4, '36302057': 14, '36264829': 11, '36264802': 9, '36228020': 10, '36227990': 4, '36201592': 8, '36173850': 9, '36173845': 7, '36108049': 7, '36108021': 6, '36074854': 4, '36074833

The output above reveals several important characteristics of our collection:

- **306 documents** have MeSH terms, while **21 documents** have none.
- **4,255 total MeSH terms** were extracted across the 306 documents, giving an average of roughly **14 terms per document**.
- **1,446 unique MeSH terms** are present. The most frequent are broad population descriptors like `Humans` (275 occurrences) and `SARS-CoV-2` (85 occurrences).

**Why this matters for retrieval:** High-frequency terms like `Humans` or `Pandemics` carry little discriminative value — they appear in almost every document. Rare, specific terms like `BNT162 Vaccine` or `Cryoelectron Microscopy` are far more useful for distinguishing relevant documents from non-relevant ones. Retrieval models that use MeSH terms (such as the expansion and ranking methods below) will naturally benefit from this specificity.

## 2. Information Retrieval with Query Expansion

With MeSH terms extracted, we can now incorporate them into the retrieval pipeline. This section implements and compares the expansion and ranking methods introduced above.

In [12]:
!pip install python-terrier

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.6 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=902cbe3a632cfd7faa18ee525b736b140a591322392388b939f6f82e04611282
  Stored in directory: /root/.cache/pip/wheels/f6/85/c2/9f0f621def52a1d5db7d29984f81e45f9fb6dfeb1a4eb6e31c
  

In [13]:
## Importing pyterrier:
import pyterrier as pt
if not pt.started():
  pt.init()

/tmp/ipykernel_2248/2134762810.py:3: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependenci…

Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar:   0%| …

Done


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_2248/2134762810.py:4: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


### 2.1 Building the Index

We index the TREC-COVID dataset using **PyTerrier** and set up a **BM25** baseline retrieval model. This will serve as our reference point for evaluating the impact of semantic enhancements.

In [14]:
# Load and format the dataset — all fields must be strings for indexing
covid_df = pd.read_csv('covid_df.csv')
covid_df = covid_df.astype('str')

covid_df['docno'] = covid_df['doc_id']
covid_df['text'] = covid_df['Title'] + " "+  covid_df['Abstract']

In [15]:
# Build the PyTerrier index
pt_index_path = './terrier_index'
indexer_df = pt.DFIndexer(pt_index_path)
indexer_df
index_df = indexer_df.index(covid_df["text"], covid_df[['docno', 'Title', 'Abstract']])

/tmp/ipykernel_2248/2846519114.py:3: DeprecationWarning: Call to deprecated class DFIndexer. (use pt.terrier.IterDictIndexer().index(dataframe.to_dict(orient='records')) instead) -- Deprecated since version 0.11.0.
  indexer_df = pt.DFIndexer(pt_index_path)


In [16]:
# Define BM25 as the baseline retrieval model
bm25 = pt.BatchRetrieve(index_df, wmodel="BM25", metadata=["docno", "Title", "Abstract"])
query = "clinical trials in covid"

bm25_result = bm25.search(query)
print("number of documents related to the query:", len(bm25_result))
bm25_result.head(3)

/tmp/ipykernel_2248/97834658.py:2: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25 = pt.BatchRetrieve(index_df, wmodel="BM25", metadata=["docno", "Title", "Abstract"])


number of documents related to the query: 196


,qid,docid,docno,Title,Abstract,rank,score,query
0,1,37,36730402,Protein decoys may battle COVID-19 and more.,Drugs designed to resemble viruses' cellular t...,0,12.361627,clinical trials in covid
1,1,153,34812653,Immune correlates analysis of the mRNA-1273 CO...,In the coronavirus efficacy (COVE) phase 3 cli...,1,10.108615,clinical trials in covid
2,1,146,34855513,Impact of community masking on COVID-19: A clu...,We conducted a cluster-randomized trial to mea...,2,7.903165,clinical trials in covid


<div style="background:#f9f9f9;border-left:5px solid #607D8B;padding:18px 22px;border-radius:6px;margin:18px 0;font-family:sans-serif;">
  <p style="margin:0 0 10px;color:#333;line-height:1.7;">
    The table above shows the <strong>top 3 documents</strong> retrieved by plain BM25 for the query <code>"clinical trials in covid"</code>. Key things to notice:
  </p>
  <ul style="color:#333;line-height:1.9;margin:0 0 10px;padding-left:20px;">
    <li> The score column represents the BM25 relevance score. Higher values indicate stronger term overlap between the query and the document. The top document ("Protein decoys may battle COVID-19 and more") scores 12.36, while the third drops to 7.9.
    <li>BM25 treats all words equally. The term <em>"clinical"</em> carries no special medical meaning here. It is just a string. MeSH-based approaches will anchor retrieval to structured medical concepts instead.</li>
  </ul>
  <p style="margin:0;color:#555;font-size:0.93em;">
    💡 <em>This BM25 result is our baseline. All improvements in later sections will be measured against it.</em>
  </p>
</div>

### 2.2 Query Expansion with Pseudo Relevance Feedback (Bo1)

Pseudo Relevance Feedback assumes the top-ranked documents from an initial retrieval are relevant, and uses them to expand the original query with new terms. Here we use the **Bo1** model from PyTerrier, which weighs expansion terms using the Bose-Einstein statistics.

The pipeline runs in two passes: an initial BM25 retrieval → Bo1 expansion → re-retrieval with the enriched query.


In [17]:
# Define the Bo1 query expansion model
bo1 = pt.rewrite.Bo1QueryExpansion(index_df)

# Two-pass retrieval pipeline: BM25 → Bo1 expansion → BM25
bm25_bo1 = bm25 >> bo1 >> bm25
bm25_bo1_res = bm25_bo1.search(query)
bm25_bo1_res.head()

,qid,docid,docno,Title,Abstract,rank,score,query_0,query
0,1,37,36730402,Protein decoys may battle COVID-19 and more.,Drugs designed to resemble viruses' cellular t...,0,11.693575,clinical trials in covid,applypipeline:off clinic^1.358862450 trial^1.8...
1,1,153,34812653,Immune correlates analysis of the mRNA-1273 CO...,In the coronavirus efficacy (COVE) phase 3 cli...,1,9.854214,clinical trials in covid,applypipeline:off clinic^1.358862450 trial^1.8...
2,1,146,34855513,Impact of community masking on COVID-19: A clu...,We conducted a cluster-randomized trial to mea...,2,8.556648,clinical trials in covid,applypipeline:off clinic^1.358862450 trial^1.8...
3,1,165,34726479,An oral SARS-CoV-2 M,The worldwide outbreak of COVID-19 caused by s...,3,6.994043,clinical trials in covid,applypipeline:off clinic^1.358862450 trial^1.8...
4,1,192,34326236,Drug-induced phospholipidosis confounds drug r...,"Repurposing drugs as treatments for COVID-19, ...",4,6.983778,clinical trials in covid,applypipeline:off clinic^1.358862450 trial^1.8...


The pipeline `bm25 >> bo1 >> bm25` works as follows:
1. The first `bm25` retrieves an initial set of documents for the query
2. `bo1` takes the top-ranked documents and extracts the most informative terms to expand the original query
3. The second `bm25` re-retrieves using the enriched query, producing a more semantically complete result set

The key parameters controlling this behavior are:
- `fb_terms` — how many new terms to add to the query (default: 10)
- `fb_docs` — how many top documents to use as feedback (default: 3)

Increasing `fb_docs` gives the model more context but risks introducing noise if early results are irrelevant. Increasing `fb_terms` broadens the query but can drift from the original intent.

In [18]:
for fb_terms_test in [5, 10, 15, 20]:
    for fb_docs_test in [3, 5, 7, 9]:
        bo1_test = pt.rewrite.Bo1QueryExpansion(index_df,fb_terms=fb_terms_test,fb_docs=fb_docs_test)
        bm25_bo1_test = bm25 >> bo1_test >> bm25
        bm25_bo1_test_res = bm25_bo1_test.search(query)
        print(f"for fb_terms={fb_terms_test},fb_docs={fb_docs_test} the result is:")
        print(bm25_bo1_test_res.head())
        print("___________________________________________________________________________")

for fb_terms=5,fb_docs=3 the result is:
  qid  docid     docno                                              Title  \
0   1     37  36730402       Protein decoys may battle COVID-19 and more.   
1   1    153  34812653  Immune correlates analysis of the mRNA-1273 CO...   
2   1    146  34855513  Impact of community masking on COVID-19: A clu...   
3   1    165  34726479                               An oral SARS-CoV-2 M   
4   1    192  34326236  Drug-induced phospholipidosis confounds drug r...   

                                            Abstract  rank      score  \
0  Drugs designed to resemble viruses' cellular t...     0  11.254209   
1  In the coronavirus efficacy (COVE) phase 3 cli...     1   9.630567   
2  We conducted a cluster-randomized trial to mea...     2   8.600024   
3  The worldwide outbreak of COVID-19 caused by s...     3   7.039843   
4  Repurposing drugs as treatments for COVID-19, ...     4   6.751069   

                    query_0                               

<div style="background:#fff8e1;border-left:5px solid #FFC107;padding:18px 22px;border-radius:6px;margin:18px 0;font-family:sans-serif;">
  <p style="margin:0 0 10px;color:#333;line-height:1.7;">
    The grid above explores how two parameters control the Bo1 expansion behaviour:
  </p>
  <ul style="color:#333;line-height:1.9;margin:0 0 12px;padding-left:20px;">
    <li><strong>fb_docs</strong> — how many top-ranked documents are treated as "pseudo-relevant" and mined for new terms.</li>
    <li><strong>fb_terms</strong> — how many expansion terms are added to the query.</li>
  </ul>
  <p style="margin:0 0 10px;color:#333;line-height:1.7;">
    <strong>Observation:</strong> Across most combinations the top-ranked document remains stable (<em>docno 36730402, "Protein decoys may battle COVID-19"</em>), which suggests the Bo1 expansion is consistently reinforcing the same high-weight terms already present in the collection. When <code>fb_docs</code> is very high (9) and <code>fb_terms</code> is large (20), the ranking can shift. This means that the expansion is pulling in less focused vocabulary.
</div>

### 2.3 Query Expansion with MeSH Terms

Instead of expanding the query with statistically frequent terms (as in Bo1), here we use **MeSH terms** from the top-ranked documents as the expansion source. This grounds the expansion in controlled medical vocabulary rather than raw term statistics, making it more reliable in the biomedical domain.

The approach follows two steps:
1. Retrieve the top documents with BM25 and look up their MeSH terms from the dictionary built in Section 1
2. Append those MeSH terms to the original query and re-retrieve

In [19]:
# Expand query using MeSH terms from the top-scoring document
docno_high_score='36730402'
mesh_terms_to_add= meshheadings_all[docno_high_score]
mesh_terms_to_add_str=','.join(str(term) for term in mesh_terms_to_add)
original_query="clinical trials in covid"
new_query=original_query +"," + mesh_terms_to_add_str
bm25_res_with_mesh= bm25.search(new_query)
bm25_res_with_mesh.head()


,qid,docid,docno,Title,Abstract,rank,score,query
0,1,37,36730402,Protein decoys may battle COVID-19 and more.,Drugs designed to resemble viruses' cellular t...,0,11.965044,"clinical trials in covid,Humans,COVID-19,Drug ..."
1,1,291,32661058,Structural basis of a shared antibody response...,Molecular understanding of neutralizing antibo...,1,9.785029,"clinical trials in covid,Humans,COVID-19,Drug ..."
2,1,274,32907861,De novo design of picomolar SARS-CoV-2 minipro...,Targeting the interaction between the severe a...,2,9.605375,"clinical trials in covid,Humans,COVID-19,Drug ..."
3,1,125,35133176,Structures of the Omicron spike trimer with AC...,The severe acute respiratory syndrome coronavi...,3,9.171797,"clinical trials in covid,Humans,COVID-19,Drug ..."
4,1,257,33154107,De novo design of potent and resilient hACE2 d...,We developed a de novo protein design strategy...,4,9.159445,"clinical trials in covid,Humans,COVID-19,Drug ..."


In [20]:
bm25.search("clinical trials in covid")

,qid,docid,docno,Title,Abstract,rank,score,query
0,1,37,36730402,Protein decoys may battle COVID-19 and more.,Drugs designed to resemble viruses' cellular t...,0,12.361627,clinical trials in covid
1,1,153,34812653,Immune correlates analysis of the mRNA-1273 CO...,In the coronavirus efficacy (COVE) phase 3 cli...,1,10.108615,clinical trials in covid
2,1,146,34855513,Impact of community masking on COVID-19: A clu...,We conducted a cluster-randomized trial to mea...,2,7.903165,clinical trials in covid
3,1,165,34726479,An oral SARS-CoV-2 M,The worldwide outbreak of COVID-19 caused by s...,3,7.711986,clinical trials in covid
4,1,192,34326236,Drug-induced phospholipidosis confounds drug r...,"Repurposing drugs as treatments for COVID-19, ...",4,7.415376,clinical trials in covid
...,...,...,...,...,...,...,...,...
191,1,139,34990216,COVID mortality in India: National survey data...,India’s national COVID death totals remain und...,191,-0.663496,clinical trials in covid
192,1,161,34735251,Children and COVID-19 in schools.,The benefits of in-person schooling with mitig...,192,-0.663825,clinical trials in covid
193,1,108,35271343,The immunology and immunopathology of COVID-19.,Considerable research effort has been made wor...,193,-0.665785,clinical trials in covid
194,1,51,36395209,COVID-19 vaccination and menstruation.,COVID-19 vaccination causes small changes to m...,194,-0.667212,clinical trials in covid


In [21]:
# Generalize the approach: expand query using MeSH terms from the top n documents
def semantic_query_expansion(ir_model, query, n_documents):
    ir_result = ir_model.search(query)
    for i in range (n_documents):
        docno_i= ir_result['docno'][i]
        if docno_i in meshheadings_all:
           mesh_terms_docno_i= meshheadings_all[docno_i]
           mesh_terms_docno_i_str=','.join(str(term) for term in mesh_terms_docno_i)
           query_v2=query +"," + mesh_terms_docno_i_str
        else:
            continue
    return query_v2


bm25.search(semantic_query_expansion(bm25, 'clinical trials in covid', 1))

,qid,docid,docno,Title,Abstract,rank,score,query
0,1,37,36730402,Protein decoys may battle COVID-19 and more.,Drugs designed to resemble viruses' cellular t...,0,11.965044,"clinical trials in covid,Humans,COVID-19,Drug ..."
1,1,291,32661058,Structural basis of a shared antibody response...,Molecular understanding of neutralizing antibo...,1,9.785029,"clinical trials in covid,Humans,COVID-19,Drug ..."
2,1,274,32907861,De novo design of picomolar SARS-CoV-2 minipro...,Targeting the interaction between the severe a...,2,9.605375,"clinical trials in covid,Humans,COVID-19,Drug ..."
3,1,125,35133176,Structures of the Omicron spike trimer with AC...,The severe acute respiratory syndrome coronavi...,3,9.171797,"clinical trials in covid,Humans,COVID-19,Drug ..."
4,1,257,33154107,De novo design of potent and resilient hACE2 d...,We developed a de novo protein design strategy...,4,9.159445,"clinical trials in covid,Humans,COVID-19,Drug ..."
...,...,...,...,...,...,...,...,...
259,1,71,35981045,T cell immunity to COVID-19 vaccines.,T cell immunity may be critical for long-term ...,259,-0.840398,"clinical trials in covid,Humans,COVID-19,Drug ..."
260,1,108,35271343,The immunology and immunopathology of COVID-19.,Considerable research effort has been made wor...,260,-0.840523,"clinical trials in covid,Humans,COVID-19,Drug ..."
261,1,161,34735251,Children and COVID-19 in schools.,The benefits of in-person schooling with mitig...,261,-0.844664,"clinical trials in covid,Humans,COVID-19,Drug ..."
262,1,51,36395209,COVID-19 vaccination and menstruation.,COVID-19 vaccination causes small changes to m...,262,-0.848973,"clinical trials in covid,Humans,COVID-19,Drug ..."


<div style="background:#f3e5f5;border-left:5px solid #9C27B0;padding:18px 22px;border-radius:6px;margin:18px 0;font-family:sans-serif;">
  <p style="margin:0 0 10px;color:#333;line-height:1.7;">
    After appending MeSH terms from the top-ranked document (<em>36730402</em>) to the original query, the second-ranked result changed from <em>"Immune correlates analysis of mRNA-1273"</em> to <em>"Structural basis of a shared antibody response"</em> (<em>docno 32661058</em>).
  </p>
  <ul style="color:#333;line-height:1.9;margin:0 0 12px;padding-left:20px;">
    <li><strong>Risk — topic drift:</strong> If the top document's MeSH terms are too specific, the expansion can pull in papers that are medically adjacent but not directly relevant to the original query intent. The <code>n_documents</code> parameter controls this trade-off.</li>
    <li><strong>Advantage over Bo1:</strong> MeSH terms are manually curated by biomedical experts — they carry richer semantic meaning than statistically frequent corpus terms selected by Bo1.</li>
  </ul>
  <p style="margin:0;color:#555;font-size:0.93em;">
    💡 <em>MeSH query expansion works best for precise clinical queries. </em>
  </p>
</div>

### 2.4 Query Expansion via MeSH Index

A document can be associated with a large number of MeSH terms, so appending all of them to the query introduces noise. Instead, we build a **separate MeSH index** and apply Bo1 pseudo relevance feedback on it to select only the most informative terms.

The MeSH index is built in three steps:
1. Attach the MeSH terms to each document in `covid_df`
2. Use the MeSH terms as the `text` field
3. Index this enriched dataset as a standalone index for expansion

In [22]:
# Build a MeSH-only dataframe
covid_df_mesh = covid_df.copy()
covid_df_mesh['mesh'] = covid_df_mesh.apply(lambda x: " ".join( meshheadings_all[str(x['doc_id'])]) if str(x['doc_id']) in meshheadings_all else ' ', axis=1  )

covid_df_mesh['docno'] = covid_df_mesh['doc_id']
covid_df_mesh['text'] = covid_df_mesh['mesh']

# Build the MeSH index
index_mesh_path = './terrier_index_mesh'
indexer_mesh = pt.DFIndexer(index_mesh_path)
index_mesh = indexer_mesh.index(covid_df_mesh["text"], covid_df_mesh[['docno', 'mesh']])


/tmp/ipykernel_2248/1014216153.py:10: DeprecationWarning: Call to deprecated class DFIndexer. (use pt.terrier.IterDictIndexer().index(dataframe.to_dict(orient='records')) instead) -- Deprecated since version 0.11.0.
  indexer_mesh = pt.DFIndexer(index_mesh_path)


10:11:21.634 [main] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (37972171) - further warnings are suppressed
10:11:24.361 [main] WARN org.terrier.structures.indexing.Indexer -- Indexed 27 empty documents


After extracting the most informative MeSH terms using Bo1 on the MeSH-only index, the expanded query is then used to retrieve documents from the original main index (Title + Abstract), not the MeSH index. This means the MeSH index serves purely as a query rewriting tool.The final ranked results come from the standard text index.

In [23]:
# Define BM25 and Bo1 expansion over the MeSH index
bo1_mesh = pt.rewrite.Bo1QueryExpansion(index_mesh)
bm25_mesh = pt.BatchRetrieve(index_mesh, wmodel="BM25", metadata=["docno"])

# Extract the expanded query from the MeSH index pipeline
query_mesh = (bm25_mesh >> bo1_mesh).search('covid clinical trials')['query'].values[0]

# Retrieve from the main index using the MeSH-expanded query
bm25.search( query_mesh )

/tmp/ipykernel_2248/1582171881.py:3: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25_mesh = pt.BatchRetrieve(index_mesh, wmodel="BM25", metadata=["docno"])


,qid,docid,docno,Title,Abstract,rank,score,query
0,1,37,36730402,Protein decoys may battle COVID-19 and more.,Drugs designed to resemble viruses' cellular t...,0,11.712654,applypipeline:off covid^1.185583383 clinic^1.6...
1,1,153,34812653,Immune correlates analysis of the mRNA-1273 CO...,In the coronavirus efficacy (COVE) phase 3 cli...,1,11.054907,applypipeline:off covid^1.185583383 clinic^1.6...
2,1,146,34855513,Impact of community masking on COVID-19: A clu...,We conducted a cluster-randomized trial to mea...,2,10.902608,applypipeline:off covid^1.185583383 clinic^1.6...
3,1,165,34726479,An oral SARS-CoV-2 M,The worldwide outbreak of COVID-19 caused by s...,3,8.702904,applypipeline:off covid^1.185583383 clinic^1.6...
4,1,125,35133176,Structures of the Omicron spike trimer with AC...,The severe acute respiratory syndrome coronavi...,4,7.893575,applypipeline:off covid^1.185583383 clinic^1.6...
...,...,...,...,...,...,...,...,...
239,1,114,35271320,Gender-responsive social protection post-COVID...,Investment in gender-responsive social protect...,239,-0.408029,applypipeline:off covid^1.185583383 clinic^1.6...
240,1,166,34709914,Defective viral RNA sensing linked to severe C...,Genetic variation in a sensor of double-strand...,240,-0.412089,applypipeline:off covid^1.185583383 clinic^1.6...
241,1,139,34990216,COVID mortality in India: National survey data...,India’s national COVID death totals remain und...,241,-0.420248,applypipeline:off covid^1.185583383 clinic^1.6...
242,1,161,34735251,Children and COVID-19 in schools.,The benefits of in-person schooling with mitig...,242,-0.420456,applypipeline:off covid^1.185583383 clinic^1.6...


## 3. Information Retrieval with Document Expansion

Rather than expanding the query, we can enrich the documents themselves by appending their MeSH terms directly to the indexed text. This way, a standard BM25 query can match documents through their associated medical concepts without any query modification.

The implementation follows three steps:
1. Build a new index where each document's text includes both its original content and its MeSH terms
2. Retrieve documents for a given query using standard BM25
3. Retrieve again with Bo1 relevance feedback and compare.


In [24]:
# Append MeSH terms to each document's text
covid_df_doc = covid_df.copy()
covid_df_doc['mesh'] = covid_df_doc.apply(lambda x: " ".join( meshheadings_all[str(x['doc_id'])]) if str(x['doc_id']) in meshheadings_all else ' ', axis=1  )
covid_df_doc['docno'] = covid_df_doc['doc_id']
covid_df_doc['text'] = covid_df_doc['Title'] + " "+  covid_df_doc['Abstract']+ " "+covid_df_doc['mesh']

# Build the document-expanded index
index_doc_path = './terrier_index_doc'
indexer_doc = pt.DFIndexer(index_doc_path)
index_doc = indexer_doc.index(covid_df_doc["text"], covid_df_doc[['docno', 'Title', 'Abstract', 'mesh']])

/tmp/ipykernel_2248/3423095639.py:9: DeprecationWarning: Call to deprecated class DFIndexer. (use pt.terrier.IterDictIndexer().index(dataframe.to_dict(orient='records')) instead) -- Deprecated since version 0.11.0.
  indexer_doc = pt.DFIndexer(index_doc_path)


In [25]:
# Define BM25 and Bo1 expansion over the document-expanded index
bo1_doc = pt.rewrite.Bo1QueryExpansion(index_doc)
bm25_doc = pt.BatchRetrieve(index_doc, wmodel="BM25", metadata=["docno", "Title", "Abstract"])

# Extract the Bo1-expanded query from the document-expanded index
query_doc = (bm25_doc >> bo1_doc).search('covid clinical trials')['query'].values[0]

# Retrieve final results using the expanded query
bm25_doc.search( query_doc )

/tmp/ipykernel_2248/1759279206.py:3: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25_doc = pt.BatchRetrieve(index_doc, wmodel="BM25", metadata=["docno", "Title", "Abstract"])


,qid,docid,docno,Title,Abstract,rank,score,query
0,1,153,34812653,Immune correlates analysis of the mRNA-1273 CO...,In the coronavirus efficacy (COVE) phase 3 cli...,0,14.484963,applypipeline:off covid^1.000000000 clinic^1.4...
1,1,165,34726479,An oral SARS-CoV-2 M,The worldwide outbreak of COVID-19 caused by s...,1,13.161984,applypipeline:off covid^1.000000000 clinic^1.4...
2,1,183,34519540,Low-dose mRNA-1273 COVID-19 vaccine generates ...,Vaccine-specific CD4,2,10.441593,applypipeline:off covid^1.000000000 clinic^1.4...
3,1,35,36730407,The NIH-led research response to COVID-19.,"Investment, collaboration, and coordination ha...",3,9.805851,applypipeline:off covid^1.000000000 clinic^1.4...
4,1,125,35133176,Structures of the Omicron spike trimer with AC...,The severe acute respiratory syndrome coronavi...,4,9.140547,applypipeline:off covid^1.000000000 clinic^1.4...
...,...,...,...,...,...,...,...,...
319,1,72,35981036,Long Covid clues emerge from patients' blood.,"Study implicates lack of key hormone, battle-w...",319,-3.740437,applypipeline:off covid^1.000000000 clinic^1.4...
320,1,166,34709914,Defective viral RNA sensing linked to severe C...,Genetic variation in a sensor of double-strand...,320,-3.740437,applypipeline:off covid^1.000000000 clinic^1.4...
321,1,66,36074854,Understanding myalgic encephalomyelitis.,Myalgic encephalomyelitis and Long Covid have ...,321,-3.811547,applypipeline:off covid^1.000000000 clinic^1.4...
322,1,16,37410823,Studies probe COVID-19 shots' link to rare sym...,Details emerge for uncommon cases of neurologi...,322,-3.818667,applypipeline:off covid^1.000000000 clinic^1.4...


## 4. Semantic Relevance Ranking

Instead of modifying queries or documents, we can incorporate semantic signals directly into the **ranking function**. This section covers two approaches:

1. **Weighting Strategy** — treat each document as a set of fields (title, abstract, MeSH terms) and assign different weights to each when scoring
2. **Learning to Rank** — train a model to combine multiple retrieval signals into an optimal ranking






### 4.1 Field-Based Weighting with BM25F

The idea here is that not all parts of a document contribute equally to relevance. A query match in the title is likely more significant than one in the abstract, and a MeSH term match signals a strong topical alignment. **BM25F** extends BM25 to support this by scoring each field separately and combining them with configurable weights.

The implementation follows three steps:
1. Define separate fields (`title`, `abstract`, `mesh`) in the dataframe
2. Index the collection with `IterDictIndexer` using the `fields` parameter
3. Retrieve using `BM25F` with field-specific weights

In [26]:
# Define separate fields for weighted indexing
covid_df_mesh['text'] = covid_df_mesh['Title'] +' ' + covid_df_mesh['Abstract'] + ' ' + covid_df_mesh['mesh']
covid_df_mesh['title'] = covid_df_mesh['Title']
covid_df_mesh['abstract'] = covid_df_mesh['Abstract']

In [27]:
iter_indexer = pt.IterDictIndexer(
    "./index_weight_fields",
    meta=['docno', 'mesh', 'title', 'abstract'],
    fields=['text', 'title', 'abstract', 'mesh'],
    text_attrs=['text', 'title', 'abstract', 'mesh']
)

index_fields = iter_indexer.index(covid_df_mesh.to_dict(orient="records"))

In [28]:
index_w = pt.IndexFactory.of(index_fields)
print(index_w.getCollectionStatistics().toString())

Number of documents: 333
Number of terms: 3685
Number of postings: 21315
Number of fields: 4
Number of tokens: 63572
Field names: [text, title, abstract, mesh]
Positions:   false



The `controls` dictionary assigns a weight to each field in order: `text`, `title`, `abstract`, `mesh`. Here we give the abstract a higher weight (`w.2 = 2`), reflecting that it carries the most descriptive content.

In [29]:
# Define field weights: text=1, title=1, abstract=2, mesh=1
controls={"w.0" : 1, "w.1" : 1, "w.2" : 2, "w.3" : 1}
bm25_fields = pt.BatchRetrieve(index_fields, wmodel="BM25F", metadata=["docno", "title", "abstract", 'mesh'], controls=controls)
bm25_fields.search('covid clinical trials').head(3)

/tmp/ipykernel_2248/19230879.py:3: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25_fields = pt.BatchRetrieve(index_fields, wmodel="BM25F", metadata=["docno", "title", "abstract", 'mesh'], controls=controls)


,qid,docid,docno,title,abstract,mesh,rank,score,query
0,1,125,35133176,Structures of the Omicron spike trimer with AC...,The severe acute respiratory syndrome coronavi...,Angiotensin-Converting Enzyme 2chemistrymetabo...,0,11.352846,covid clinical trials
1,1,37,36730402,Protein decoys may battle COVID-19 and more.,Drugs designed to resemble viruses' cellular t...,Humans COVID-19 Drug Design COVID-19 Drug Trea...,1,9.879011,covid clinical trials
2,1,153,34812653,Immune correlates analysis of the mRNA-1273 CO...,In the coronavirus efficacy (COVE) phase 3 cli...,2019-nCoV Vaccine mRNA-1273immunology Adolesce...,2,7.845609,covid clinical trials


We can also inspect the BM25 score contributed by each field individually using `SingleFieldModel`. This helps understand which fields are driving relevance for a given query.

In [30]:
# Retrieve scores broken down by individual field
bm25f0 = pt.BatchRetrieve(index_fields, wmodel='SingleFieldModel(BM25,0)', metadata=["docno", "title", "abstract", 'mesh'])
bm25f1 = pt.BatchRetrieve(index_fields, wmodel='SingleFieldModel(BM25,1)', metadata=["docno", "title", "abstract", 'mesh'])
bm25f2 = pt.BatchRetrieve(index_fields, wmodel='SingleFieldModel(BM25,2)', metadata=["docno", "title", "abstract", 'mesh'])
bm25f3 = pt.BatchRetrieve(index_fields, wmodel='SingleFieldModel(BM25,3)', metadata=["docno", "title", "abstract", 'mesh'])


bm25f0.search('covid clinical trials').head(3)

/tmp/ipykernel_2248/843491348.py:2: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25f0 = pt.BatchRetrieve(index_fields, wmodel='SingleFieldModel(BM25,0)', metadata=["docno", "title", "abstract", 'mesh'])
/tmp/ipykernel_2248/843491348.py:3: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25f1 = pt.BatchRetrieve(index_fields, wmodel='SingleFieldModel(BM25,1)', metadata=["docno", "title", "abstract", 'mesh'])
/tmp/ipykernel_2248/843491348.py:4: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25f2 = pt.BatchRetrieve(index_fields, wmodel='SingleFieldModel(BM25,2)', metadata=["docno", "title", "abstract", 'mesh'])
/tmp/ipykernel_2248/843491348.py:5: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Ret

,qid,docid,docno,title,abstract,mesh,rank,score,query
0,1,125,35133176,Structures of the Omicron spike trimer with AC...,The severe acute respiratory syndrome coronavi...,Angiotensin-Converting Enzyme 2chemistrymetabo...,0,6.911068,covid clinical trials
1,1,153,34812653,Immune correlates analysis of the mRNA-1273 CO...,In the coronavirus efficacy (COVE) phase 3 cli...,2019-nCoV Vaccine mRNA-1273immunology Adolesce...,1,6.307767,covid clinical trials
2,1,165,34726479,An oral SARS-CoV-2 M,The worldwide outbreak of COVID-19 caused by s...,"Administration, Oral Animals COVID-19virology ...",2,5.783207,covid clinical trials


We can combine all field scores into a single pipeline using the `**` operator, which stacks the individual field scores into a `features` column alongside the final BM25F score. This gives a clear view of each field's contribution per retrieved document.

In [31]:
# Pipeline: BM25F ranking followed by per-field score breakdown
models_pipeline = bm25_fields >> (bm25f0
                        ** bm25f1
                        ** bm25f2
                        ** bm25f3)
result=models_pipeline.search('covid clinical trials')
result[['docno', 'score', 'features']]

,docno,score,features
0,35133176,11.352846,"[6.911067580785076, 0.0, 6.824372658439044, 0.0]"
1,36730402,9.879011,"[4.815477773198886, -3.882269973439655, 12.739..."
2,34812653,7.845609,"[6.307766866835206, 4.0110853676742515, 1.4687..."
3,34726479,7.583894,"[5.783207357716604, 0.0, 2.9771971647273765, 4..."
4,36730407,7.423219,"[5.123061676482256, -3.882269973439655, 0.0, 4..."
...,...,...,...
303,35271331,-7.495669,"[-6.619154236863066, 0.0, -5.508602277494326, ..."
304,35981045,-7.526136,"[-6.875698762058245, -4.123728522641694, -5.40..."
305,36395209,-7.531876,"[-6.875698762058245, -4.397214125260795, -5.45..."
306,35482858,-7.542353,"[-6.349079353260471, 0.0, -5.620421577666367, ..."


<div style="background:#e8f5e9;border-left:5px solid #4CAF50;padding:18px 22px;border-radius:6px;margin:18px 0;font-family:sans-serif;">
  <p style="margin:0 0 10px;color:#333;line-height:1.7;">
    The <code>features</code> column contains a 4-element vector: <code>[text, title, abstract, mesh]</code> scores. For the top document (<em>35133176</em>) the vector is approximately <code>[6.91, 0.0, 6.82, 0.0]</code>:
  </p>
  <ul style="color:#333;line-height:1.9;margin:0 0 12px;padding-left:20px;">
    <li><strong>text = 6.91, abstract = 6.82:</strong> The query terms appear strongly in both the full text and the abstract.</li>
    <li><strong>title = 0.0, mesh = 0.0:</strong> The query terms do not appear in the title or MeSH tags of this document.</li>

  </ul>
</div>

### 4.2 Learning to Rank

Instead of manually tuning field weights, **Learning to Rank (LTR)** trains a model to find the optimal combination of retrieval signals automatically. Each document is represented by a feature vector and a supervised model learns to rank documents by relevance.

To train the model we need labeled data: query-document pairs with known relevance judgments. We use the **TREC-COVID** test collection for this, which provides official relevance assessments that serve as our ground truth.


[TREC-COVID](https://ir.nist.gov/trec-covid/) is a benchmark test collection built in 2020, comprising nearly 200,000 scientific publications on COVID-19. It includes a set of queries and official relevance judgments, making it a standard evaluation resource for biomedical IR systems.

In [32]:
import json
import numpy as np

In [33]:
# Load TREC-COVID topics (queries) and relevance judgments
dataset = pt.datasets.get_dataset('irds:cord19/trec-covid')
topics = dataset.get_topics('title')
qrels = dataset.get_qrels()

[INFO] [starting] https://ir.nist.gov/covidSubmit/data/topics-rnd5.xml
[INFO] [finished] https://ir.nist.gov/covidSubmit/data/topics-rnd5.xml: [00:00] [18.7kB] [28.4MB/s]
[INFO] [starting] https://ir.nist.gov/covidSubmit/data/qrels-covid_d5_j0.5-5.txt
[INFO] [finished] https://ir.nist.gov/covidSubmit/data/qrels-covid_d5_j0.5-5.txt: [00:00] [1.14MB] [8.65MB/s]


In [34]:
# Load the pre-built PyTerrier index for TREC-COVID
trec_covid_index = pt.IndexFactory.of(
    pt.get_dataset('trec-covid').get_index('terrier_stemmed_positions')
)

data.meta.zdata:   0%|          | 0.00/4.38M [00:00<?, ?iB/s]

data.meta.idx:   0%|          | 0.00/1.46M [00:00<?, ?iB/s]

data.properties:   0%|          | 0.00/4.37k [00:00<?, ?iB/s]

data.meta-0.fsomapfile:   0%|          | 0.00/5.29M [00:00<?, ?iB/s]

data.inverted.bf:   0%|          | 0.00/48.7M [00:00<?, ?iB/s]

...

data.lexicon.fsomapid:   0%|          | 0.00/619k [00:00<?, ?iB/s]

data.lexicon.fsomaphash:   0%|          | 0.00/0.99k [00:00<?, ?iB/s]

md5sums:   0%|          | 0.00/537 [00:00<?, ?iB/s]

data.direct.bf:   0%|          | 0.00/50.6M [00:00<?, ?iB/s]

data.document.fsarrayfile:   0%|          | 0.00/4.56M [00:00<?, ?iB/s]

#### Defining the Feature Pipeline

We define five retrieval signals as features for the LTR model. Each document will be scored by all five models, and the resulting scores form the feature vector used for training.

In [35]:
# Define individual retrieval models on the TREC-COVID index
tc_bm25 = pt.BatchRetrieve(trec_covid_index, wmodel="BM25")
tc_tfidf = pt.BatchRetrieve(trec_covid_index, wmodel="TF_IDF")
tc_tf = pt.BatchRetrieve(trec_covid_index, wmodel="Tf")
qe = pt.rewrite.Bo1QueryExpansion(trec_covid_index)

/tmp/ipykernel_2248/206041410.py:2: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  tc_bm25 = pt.BatchRetrieve(trec_covid_index, wmodel="BM25")
/tmp/ipykernel_2248/206041410.py:3: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  tc_tfidf = pt.BatchRetrieve(trec_covid_index, wmodel="TF_IDF")
/tmp/ipykernel_2248/206041410.py:4: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  tc_tf = pt.BatchRetrieve(trec_covid_index, wmodel="Tf")


In [36]:
# Build the feature pipeline: BM25 candidate retrieval followed by all five feature scores
ltr_feats1 = tc_bm25  >> (
    tc_bm25                         # Feature 1: BM25
    **
    (tc_bm25 >> qe >> tc_bm25)      # Feature 2: BM25 with Bo1 expansion
    **
    (tc_tf)                         # Feature 3: TF
    **
    (tc_tfidf)                      # Feature 4: TF-IDF
    **
    (tc_tfidf >> qe >> tc_tfidf)    # Feature 5: TF-IDF with Bo1 expansion
)

fnames=["BM25", "BM25_bo1", 'TF', 'TFIDF', "TFIDF_bo1" ]

In [37]:
# Inspect the feature vectors for a sample query
ltr_feats1.search("coronovirus")

10:11:44.293 [main] WARN org.terrier.structures.FSADocumentIndex -- This index has fields, but FSADocumentIndex is used (which stores fields lengths on disk); If using field-based models such as BM25F, change to index.document.class in the index  properties file to FSAFieldDocumentIndex or FSADocumentIndexInMemFields to support efficient retrieval. If you don't use (e.g.) BM25F, this warning can be ignored


/usr/local/lib/python3.12/dist-packages/pyterrier/_ops.py:264: UserWarning: Got number of results different expected from (TerrierRetr(BM25) >> QueryExpansion(/root/.pyterrier/corpora/trec-covid/index/terrier_stemmed_positions/data.properties,3,10,<org.terrier.querying.QueryExpansion at 0x7db2f81ebc00 jclass=org/terrier/querying/QueryExpansion jself=<LocalRef obj=0x2368eba0 at 0x7db2f839eb50>>) >> TerrierRetr(BM25)), expected 36 received 1000, feature scores for any missing documents be 0, extraneous documents will be removed
  warn(
/usr/local/lib/python3.12/dist-packages/pyterrier/_ops.py:264: UserWarning: Got number of results different expected from (TerrierRetr(TF_IDF) >> QueryExpansion(/root/.pyterrier/corpora/trec-covid/index/terrier_stemmed_positions/data.properties,3,10,<org.terrier.querying.QueryExpansion at 0x7db2f81ebc00 jclass=org/terrier/querying/QueryExpansion jself=<LocalRef obj=0x2368eba0 at 0x7db2f839eb50>>) >> TerrierRetr(TF_IDF)), expected 36 received 1000, feature 

,qid,docid,docno,rank,score,query,query_0,features
0,1,144734,6jvqsvy5,0,20.354386,coronovirus,coronovirus,"[20.35438619404753, 20.35438619404753, 1.0, 11..."
1,1,13691,tkxfzhwh,1,19.958371,coronovirus,coronovirus,"[19.958371092978137, 19.958371092978137, 1.0, ..."
2,1,54405,c67kgvr8,2,19.958371,coronovirus,coronovirus,"[19.958371092978137, 19.958371092978137, 1.0, ..."
3,1,187070,8bn74f2z,3,19.329562,coronovirus,coronovirus,"[19.329562397928484, 19.329562397928484, 2.0, ..."
4,1,162708,3n9f84hg,4,18.973951,coronovirus,coronovirus,"[18.973951486762303, 18.973951486762303, 1.0, ..."
5,1,84009,0seoiqre,5,18.515491,coronovirus,coronovirus,"[18.515490662809988, 18.515490662809988, 2.0, ..."
6,1,92845,ciqs6l7e,6,17.666861,coronovirus,coronovirus,"[17.666860555114013, 17.666860555114013, 1.0, ..."
7,1,92846,qe9w4qbu,7,17.666861,coronovirus,coronovirus,"[17.666860555114013, 17.666860555114013, 1.0, ..."
8,1,151848,egzztatj,8,17.367750,coronovirus,coronovirus,"[17.367750105171087, 17.367750105171087, 1.0, ..."
9,1,113996,a47onmje,9,17.077076,coronovirus,coronovirus,"[17.07707581862232, 17.07707581862232, 2.0, 9...."


In [38]:
# Split topics into train (60%), validation (20%), and test (20%) sets
train_topics, valid_topics, test_topics = np.split(topics, [int(.6*len(topics)), int(.8*len(topics))])

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [39]:
# Train a Random Forest LTR model on the feature pipeline
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=400, verbose=1)

rf_pipe = ltr_feats1 >> pt.ltr.apply_learned_model(rf)

%time rf_pipe.fit(train_topics, qrels)

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    8.6s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:   31.7s


CPU times: user 1min 22s, sys: 344 ms, total: 1min 22s
Wall time: 1min 20s


[Parallel(n_jobs=1)]: Done 400 out of 400 | elapsed:  1.1min finished


In [40]:
# Evaluate BM25 baseline vs LTR model on the test set using recall
results = pt.pipelines.Experiment([tc_bm25, rf_pipe], test_topics, qrels, ["recall"], names=["BM25 Baseline", "LTR method"])
results

/tmp/ipykernel_2248/196302259.py:2: DeprecationWarning: Call to deprecated function (or staticmethod) Experiment. (use pt.Experiment() instead) -- Deprecated since version 1.0.0.
  results = pt.pipelines.Experiment([tc_bm25, rf_pipe], test_topics, qrels, ["recall"], names=["BM25 Baseline", "LTR method"])
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done 400 out of 400 | elapsed:    0.4s finished


,name,R@5,R@10,R@15,R@20,R@30,R@100,R@200,R@500,R@1000
0,BM25 Baseline,0.012742,0.024938,0.035193,0.044638,0.067530,0.166752,0.269120,0.404296,0.512072
1,LTR method,0.006898,0.013649,0.021589,0.029409,0.045144,0.139608,0.250118,0.419478,0.490882


<div style="background:#fce4ec;border-left:5px solid #E91E63;padding:18px 22px;border-radius:6px;margin:18px 0;font-family:sans-serif;">
  <p style="margin:0 0 10px;color:#333;line-height:1.7;">
    The evaluation uses <strong>Recall@K</strong> that shows the fraction of all known relevant documents that appear in the top-K results. Higher is better.
  </p>
  <p style="margin:0 0 10px;color:#333;line-height:1.7;">
    The LTR model scores <em>lower</em> than plain BM25 across all cutoffs. This result may seem surprising and can be due to the small size of training set. TREC-COVID has only 50 topics total. With a 60/20/20 split that leaves ~30 training queries. A 400-tree Random Forest with 5 features will overfit on such a small sample.
  </p>
  </ul>

</div>



## 5. Conclusion

  <p>This project was about making search engines smarter when looking for medical research papers.
  Instead of just matching words, the goal was to make the system understand medical concepts.</p>

  <p>We started with <strong>BM25</strong>, the standard way search engines work. It just counts
  how many times your search words appear in a document.</p>

  <p>To improve the results, we tried several approaches:

<strong>1. Query Expansion with Bo1:</strong>
After an initial search, it looks at the top results, pulls out the most common useful words, adds them to the query, and searches again.

<strong>2. Query Expansion with MeSH Terms:</strong>
It has the same idea, but instead of random frequent words, it adds expert medical tags (MeSH terms) from the top documents. More meaningful for medical searches, but risks pulling in unrelated topics.

<strong>3. Query Expansion via MeSH Index:</strong>
This is a cleaner version of the above. It builds an index of just MeSH terms, uses Bo1 to pick only the most relevant ones, then expands the query with those. So it has less noise.

<strong>4. Document Expansion with MeSH:</strong>
Instead of touching the query, it adds MeSH tags directly to each document before indexing. At search time, BM25 just naturally matches more concepts.

<strong>5. BM25F (Weighted Fields):</strong>
Treats title, abstract, and MeSH tags as separate fields and scores them with different weights. A match in the title counts more than one buried in the abstract.

<strong>6. Learning to Rank (LTR):</strong>
Trains a machine learning model to combine multiple search signals (BM25, TF-IDF, Bo1, etc.) into one optimal score. In this project it actually performed worse than plain BM25. It is likely due to the small size of dataset.


Overall, there is no single best method — it depends on your situation. If you want something simple and reliable, document expansion with MeSH is a great choice since it requires no changes at search time. If your queries are very specific medical questions, MeSH-based query expansion works better. If you have a large labeled dataset, Learning to Rank can outperform all other methods by combining every signal together. In short: start simple, and only add complexity when your data supports it.

</div>